In [1]:
!pip install -U crewai crewai-tools litellm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

### 1. Installation
In the cell above, we installed the core libraries: `crewai` for the agentic framework, `crewai-tools` for extra capabilities, and `litellm` to handle different model providers.

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

### 2. Environment Setup
We securely retrieved the Groq API key from Colab's user secrets and set it as an environment variable to authenticate our AI agents.

In [3]:
from crewai import LLM

llm = LLM(
    model="groq/openai/gpt-oss-120b",
    temperature=0.7
)

### 3. LLM Configuration
We initialized the LLM (Large Language Model) using Groq's endpoint. We started with the `gpt-oss-120b` model for high-quality content generation.

In [4]:
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

### 4. Cache Management
We applied a small patch to the CrewAI cache system to ensure consistent behavior during development in the notebook environment.

In [5]:
from crewai import Agent
manager_llm = LLM(
    model="groq/openai/gpt-oss-120b",
    temperature=0.7
)

manager = Agent(
    role="Senior Content Project Manager",
    goal="Analyze the source content and delegate platform-specific repurposing tasks to writer agents, ensuring each output matches the platform's tone and format",
    backstory=(
        "You are a seasoned content strategist with years of experience "
        "managing content teams. You understand what works on LinkedIn vs Twitter/X vs "
        "Instagram, and you brief your writers clearly so their output is on-brand and platform-appropriate."
    ),
    llm=manager_llm,
    allow_delegation=True,
    verbose=True
)

### 5. Manager Agent Definition
We defined the Manager agent. This agent is responsible for coordinating the workflow and ensuring the final report is high quality.

In [6]:
linkedin_writer = Agent(
    role="LinkedIn Content Writer",
    goal="Turn the source content into a professional, engaging LinkedIn post with a hook, value, and a call-to-action",
    backstory=(
        "You are an expert LinkedIn ghostwriter who knows how to write posts that "
        "get engagement — short paragraphs, a strong opening line, and relevant hashtags."
    ),
    llm=llm,
    verbose=True
)

twitter_writer = Agent(
    role="Twitter/X Thread Writer",
    goal="Turn the source content into a concise, punchy Twitter/X thread (3-6 tweets)",
    backstory=(
        "You are a Twitter/X growth expert who writes threads that hook readers "
        "in the first tweet and keep them scrolling."
    ),
    llm=llm,
    verbose=True
)

instagram_writer = Agent(
    role="Instagram Caption Writer",
    goal="Turn the source content into a catchy Instagram caption with relevant hashtags and emojis",
    backstory=(
        "You are a social media copywriter specializing in Instagram captions that "
        "are visually scannable, emotionally engaging, and hashtag-optimized."
    ),
    llm=llm,
    verbose=True
)

### 6. Specialized Writer Agents
We created three specialized agents: one for LinkedIn (professional), one for Twitter (punchy threads), and one for Instagram (catchy captions with emojis).

In [7]:
from crewai import Task

source_content = """
Why Small Businesses Should Start Using AI Today

Artificial intelligence is no longer just for big tech companies with huge budgets.
Over the past two years, AI tools have become affordable, accessible, and genuinely
useful for small businesses — from automating customer support with chatbots, to
generating marketing content in minutes, to analyzing sales data without hiring a
data analyst.

The biggest myth is that AI adoption requires technical expertise. In reality, most
modern AI tools are built with simple, no-code interfaces designed for non-technical
users. A small bakery can use AI to schedule social media posts. A local gym can use
AI to answer common customer questions instantly, 24/7. A freelance consultant can
use AI to draft proposals in a fraction of the time.

The businesses that adopt AI early aren't just saving time — they're freeing up
mental energy to focus on what actually grows a business: relationships, creativity,
and strategy. The tools will only get cheaper and smarter. The question isn't whether
to adopt AI, but how soon you start.
"""

linkedin_task = Task(
    description=f"""Rewrite the blog content as a LinkedIn post.
    Source Content: {source_content}
    - Professional, thought-leadership tone
    - 150-250 words
    - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols like * or -)
    - Use line breaks (blank lines) for readability instead of headers
    - MANDATORY: The post MUST end with a genuine question (not a statement or CTA) —
      verify the last sentence is a question before finalizing
    - Add 3-5 professional hashtags at the very end, space-separated (e.g. #AIEngineering #TechCareers)
    - Do NOT reuse blog's exact headings or sentence structure
    - CRITICAL: Output must be RAW plain text only — do NOT wrap the post in quotation marks,
      do NOT prefix it with "Post:" or any label. Start directly with the hook line
    - Output must be ready to copy-paste directly into LinkedIn — no markdown syntax""",
    expected_output="Plain text LinkedIn post (no markdown, no quotes, no labels), 150-250 words, ending with a question, then hashtags.",
    agent=linkedin_writer,
    async_execution=True,
    context=[]
)

twitter_task = Task(
    description=f"""Convert the blog into a Twitter/X thread.
    Source Content: {source_content}
    - CRITICAL: Each tweet's text (excluding numbering/hashtags) MUST be 35 words or fewer — count before finalizing
    - Punchy, conversational tone — short sentences
    - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)
    - Use thread numbering like "1/5" at the start of each tweet
    - CRITICAL: Distribute hashtags ACROSS the thread — do NOT dump all hashtags on the last tweet.
      Each tweet should carry 0-2 of its own hashtags, spread naturally through the thread
    - Rephrase content in your own words
    - Separate each tweet clearly with a blank line so it's easy to copy individually""",
    expected_output="Plain text Twitter thread, numbered tweets separated by blank lines, each under 35 words, hashtags distributed across tweets not stacked on one.",
    agent=twitter_writer,
    async_execution=True,
    context=[]
)

instagram_task = Task(
    description=f"""Write an Instagram caption based on the blog.
    Source Content: {source_content}
    - Casual, engaging, hook-driven opening line
    - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)
    - Short paragraphs (2-3 lines max) separated by blank lines, heavy but tasteful emoji use
    - End with a call-to-action (e.g. 'Save this for later 🔖')
    - CRITICAL: Use 8-15 Instagram-native hashtags that are DIFFERENT from typical LinkedIn
      professional hashtags — favor casual/community tags (e.g. #SmallBizTips #AIforBusiness
      #EntrepreneurLife #WorkSmarter) over formal ones (e.g. avoid #AIEngineering, #TechCareers style tags)
    - Must read distinctly different from LinkedIn/Twitter versions in both tone AND hashtags
    - Output must be ready to copy-paste directly into Instagram — no markdown syntax""",
    expected_output="Plain text Instagram caption (no markdown), short paragraphs, ending with hashtags that differ from LinkedIn's hashtags.",
    agent=instagram_writer,
    async_execution=False,
    context=[]
)

### 7. Task Definition
We provided the source article about AI for small businesses and defined four tasks: three asynchronous generation tasks and one final compilation task performed by the manager.

In [8]:
from crewai import Crew, Process

# Simplified crew without manager delegation to avoid TPM spikes
content_crew = Crew(
    agents=[linkedin_writer, twitter_writer, instagram_writer],
    tasks=[linkedin_task, twitter_task, instagram_task],
    process=Process.sequential,
    verbose=True
)

### 8. Crew Assembly
We assembled the writers and the manager into a `Crew`, setting up a sequential process where agents collaborate to produce the final social media kit.

In [9]:
result = await content_crew.kickoff_async()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 219acca6-b826-4911-988a-de371e9fa899                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Rewrite the blog content as a LinkedIn post.                                                             │
│      Source Content:                                                                                            │
│  Why Small Businesses Should Start Using AI Today                                                               │
│                                                                                                                 │
│  Artificial intelligence is no longer just for big tech companies with huge budgets.                            │
│  Over the past two years, AI tools have become affordable, accessible, and genuinely                            │
│  useful for small businesses — from automating customer support with chatbots, to                               │
│  generating marketing content in minutes, to analyzing sales data without hiring a                              │
│  data analyst.                                                                                                  │
│                                                                                                                 │
│  The biggest myth is that AI adoption requires technical expertise. In reality, most                            │
│  modern AI tools are built with simple, no-code interfaces designed for non-technical                           │
│  users. A small bakery can use AI to schedule social media posts. A local gym can use                           │
│  AI to answer common customer questions instantly, 24/7. A freelance consultant can                             │
│  use AI to draft proposals in a fraction of the time.                                                           │
│                                                                                                                 │
│  The businesses that adopt AI early aren't just saving time — they're freeing up                                │
│  mental energy to focus on what actually grows a business: relationships, creativity,                           │
│  and strategy. The tools will only get cheaper and smarter. The question isn't whether                          │
│  to adopt AI, but how soon you start.                                                                           │
│                                                                                                                 │
│      - Professional, thought-leadership tone                                                                    │
│      - 150-250 words                                                                                            │
│      - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols like * or -)                   │
│      - Use line breaks (blank lines) for readability instead of headers                                         │
│      - MANDATORY: The post MUST end with a genuine question (not a statement or CTA) —                          │
│        verify the last sentence is a question before finalizing                                                 │
│      - Add 3-5 professional hashtags at the very end, space-separated (e.g. #AIEngineering #TechCareers)        │
│      - Do NOT reuse blog's exact headings or sentence structure                                                 │
│      - CRITICAL: Output must be RAW plain text only — do NOT wrap the post in quotation marks,                  │
│        do NOT prefix it with "Post:" or any label. Start directly with the hook line                            │
│      - Output must be ready to copy-paste directly into

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Convert the blog into a Twitter/X thread.                                                                │
│      Source Content:                                                                                            │
│  Why Small Businesses Should Start Using AI Today                                                               │
│                                                                                                                 │
│  Artificial intelligence is no longer just for big tech companies with huge budgets.                            │
│  Over the past two years, AI tools have become affordable, accessible, and genuinely                            │
│  useful for small businesses — from automating customer support with chatbots, to                               │
│  generating marketing content in minutes, to analyzing sales data without hiring a                              │
│  data analyst.                                                                                                  │
│                                                                                                                 │
│  The biggest myth is that AI adoption requires technical expertise. In reality, most                            │
│  modern AI tools are built with simple, no-code interfaces designed for non-technical                           │
│  users. A small bakery can use AI to schedule social media posts. A local gym can use                           │
│  AI to answer common customer questions instantly, 24/7. A freelance consultant can                             │
│  use AI to draft proposals in a fraction of the time.                                                           │
│                                                                                                                 │
│  The businesses that adopt AI early aren't just saving time — they're freeing up                                │
│  mental energy to focus on what actually grows a business: relationships, creativity,                           │
│  and strategy. The tools will only get cheaper and smarter. The question isn't whether                          │
│  to adopt AI, but how soon you start.                                                                           │
│                                                                                                                 │
│      - CRITICAL: Each tweet's text (excluding numbering/hashtags) MUST be 35 words or fewer — count before      │
│  finalizing                                                                                                     │
│      - Punchy, conversational tone — short sentences                                                            │
│      - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)                               │
│      - Use thread numbering like "1/5" at the start of each tweet                                               │
│      - CRITICAL: Distribute hashtags ACROSS the thread — do NOT dump all hashtags on the last tweet.            │
│        Each tweet should carry 0-2 of its own hashtags, spread naturally through the thread                     │
│      - Rephrase content in your own words                                                                       │
│      - Separate each tweet clearly with a blank line so it's easy to copy individually                          │
│  ID: daf9c5dc-e9a9-4b76-9486-588343f6962a                                                                       │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: LinkedIn Content Writer                                                                                 │
│                                                                                                                 │
│  Task: Rewrite the blog content as a LinkedIn post.                                                             │
│      Source Content:                                                                                            │
│  Why Small Businesses Should Start Using AI Today                                                               │
│                                                                                                                 │
│  Artificial intelligence is no longer just for big tech companies with huge budgets.                            │
│  Over the past two years, AI tools have become affordable, accessible, and genuinely                            │
│  useful for small businesses — from automating customer support with chatbots, to                               │
│  generating marketing content in minutes, to analyzing sales data without hiring a                              │
│  data analyst.                                                                                                  │
│                                                                                                                 │
│  The biggest myth is that AI adoption requires technical expertise. In reality, most                            │
│  modern AI tools are built with simple, no-code interfaces designed for non-technical                           │
│  users. A small bakery can use AI to schedule social media posts. A local gym can use                           │
│  AI to answer common customer questions instantly, 24/7. A freelance consultant can                             │
│  use AI to draft proposals in a fraction of the time.                                                           │
│                                                                                                                 │
│  The businesses that adopt AI early aren't just saving time — they're freeing up                                │
│  mental energy to focus on what actually grows a business: relationships, creativity,                           │
│  and strategy. The tools will only get cheaper and smarter. The question isn't whether                          │
│  to adopt AI, but how soon you start.                                                                           │
│                                                                                                                 │
│      - Professional, thought-leadership tone                                                                    │
│      - 150-250 words                                                                                            │
│      - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols like * or -)                   │
│      - Use line breaks (blank lines) for readability instead of headers                                         │
│      - MANDATORY: The post MUST end with a genuine question (not a statement or CTA) —                          │
│        verify the last sentence is a question before finalizing                                                 │
│      - Add 3-5 professional hashtags at the very end, space-separated (e.g. #AIEngineering #TechCareers)        │
│      - Do NOT reuse blog's exact headings or sentence structure                                                 │
│      - CRITICAL: Output must be RAW plain text only — do NOT wrap the post in quotation marks,                  │
│        do NOT prefix it with "Post:" or any label. Star

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Twitter/X Thread Writer                                                                                 │
│                                                                                                                 │
│  Task: Convert the blog into a Twitter/X thread.                                                                │
│      Source Content:                                                                                            │
│  Why Small Businesses Should Start Using AI Today                                                               │
│                                                                                                                 │
│  Artificial intelligence is no longer just for big tech companies with huge budgets.                            │
│  Over the past two years, AI tools have become affordable, accessible, and genuinely                            │
│  useful for small businesses — from automating customer support with chatbots, to                               │
│  generating marketing content in minutes, to analyzing sales data without hiring a                              │
│  data analyst.                                                                                                  │
│                                                                                                                 │
│  The biggest myth is that AI adoption requires technical expertise. In reality, most                            │
│  modern AI tools are built with simple, no-code interfaces designed for non-technical                           │
│  users. A small bakery can use AI to schedule social media posts. A local gym can use                           │
│  AI to answer common customer questions instantly, 24/7. A freelance consultant can                             │
│  use AI to draft proposals in a fraction of the time.                                                           │
│                                                                                                                 │
│  The businesses that adopt AI early aren't just saving time — they're freeing up                                │
│  mental energy to focus on what actually grows a business: relationships, creativity,                           │
│  and strategy. The tools will only get cheaper and smarter. The question isn't whether                          │
│  to adopt AI, but how soon you start.                                                                           │
│                                                                                                                 │
│      - CRITICAL: Each tweet's text (excluding numbering/hashtags) MUST be 35 words or fewer — count before      │
│  finalizing                                                                                                     │
│      - Punchy, conversational tone — short sentences                                                            │
│      - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)                               │
│      - Use thread numbering like "1/5" at the start of each tweet                                               │
│      - CRITICAL: Distribute hashtags ACROSS the thread — do NOT dump all hashtags on the last tweet.            │
│        Each tweet should carry 0-2 of its own hashtags, spread naturally through the thread                     │
│      - Rephrase content in your own words                                                                       │
│      - Separate each tweet clearly with a blank line so it's easy to copy individually                          │
│                                                        

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Instagram Caption Writer                                                                                │
│                                                                                                                 │
│  Task: Write an Instagram caption based on the blog.                                                            │
│      Source Content:                                                                                            │
│  Why Small Businesses Should Start Using AI Today                                                               │
│                                                                                                                 │
│  Artificial intelligence is no longer just for big tech companies with huge budgets.                            │
│  Over the past two years, AI tools have become affordable, accessible, and genuinely                            │
│  useful for small businesses — from automating customer support with chatbots, to                               │
│  generating marketing content in minutes, to analyzing sales data without hiring a                              │
│  data analyst.                                                                                                  │
│                                                                                                                 │
│  The biggest myth is that AI adoption requires technical expertise. In reality, most                            │
│  modern AI tools are built with simple, no-code interfaces designed for non-technical                           │
│  users. A small bakery can use AI to schedule social media posts. A local gym can use                           │
│  AI to answer common customer questions instantly, 24/7. A freelance consultant can                             │
│  use AI to draft proposals in a fraction of the time.                                                           │
│                                                                                                                 │
│  The businesses that adopt AI early aren't just saving time — they're freeing up                                │
│  mental energy to focus on what actually grows a business: relationships, creativity,                           │
│  and strategy. The tools will only get cheaper and smarter. The question isn't whether                          │
│  to adopt AI, but how soon you start.                                                                           │
│                                                                                                                 │
│      - Casual, engaging, hook-driven opening line                                                               │
│      - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)                               │
│      - Short paragraphs (2-3 lines max) separated by blank lines, heavy but tasteful emoji use                  │
│      - End with a call-to-action (e.g. 'Save this for later 🔖')                                                │
│      - CRITICAL: Use 8-15 Instagram-native hashtags that are DIFFERENT from typical LinkedIn                    │
│        professional hashtags — favor casual/community tags (e.g. #SmallBizTips #AIforBusiness                   │
│        #EntrepreneurLife #WorkSmarter) over formal ones (e.g. avoid #AIEngineering, #TechCareers style tags)    │
│      - Must read distinctly different from LinkedIn/Twitter versions in both tone AND hashtags                  │
│      - Output must be ready to copy-paste directly into Instagram — no markdown syntax                          │
│                                                         

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Instagram Caption Writer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  🚀 Ready to give your small biz a super‑charged boost?                                                         │
│                                                                                                                 │
│  AI isn’t just for the tech giants anymore—affordable tools are here, and they work wonders for local shops.    │
│                                                                                                                 │
│  No‑code, user‑friendly platforms mean you don’t need a PhD to automate. A bakery can schedule posts, a gym     │
│  can answer questions 24/7, and a freelancer can draft proposals in minutes.                                    │
│                                                                                                                 │
│  Early adopters aren’t just saving time; they’re freeing mental energy to focus on relationships, creativity,   │
│  and strategy.                                                                                                  │
│                                                                                                                 │
│  The tools will only get cheaper and smarter—so the real question is: how soon will you start?                  │
│                                                                                                                 │
│  Save this for later 🔖                                                                                         │
│                                                                                                                 │
│  #SmallBizTips #AIforBiz #EntrepreneurLife #WorkSmarter #BizGrowth #DigitalSidekick #CreativeHustle #ShopLocal  │
│  #BizHack #TechMadeEasy #AIForAll #StartupVibes #MarketingMagic #TimeSaver                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an Instagram caption based on the blog.                                                            │
│      Source Content:                                                                                            │
│  Why Small Businesses Should Start Using AI Today                                                               │
│                                                                                                                 │
│  Artificial intelligence is no longer just for big tech companies with huge budgets.                            │
│  Over the past two years, AI tools have become affordable, accessible, and genuinely                            │
│  useful for small businesses — from automating customer support with chatbots, to                               │
│  generating marketing content in minutes, to analyzing sales data without hiring a                              │
│  data analyst.                                                                                                  │
│                                                                                                                 │
│  The biggest myth is that AI adoption requires technical expertise. In reality, most                            │
│  modern AI tools are built with simple, no-code interfaces designed for non-technical                           │
│  users. A small bakery can use AI to schedule social media posts. A local gym can use                           │
│  AI to answer common customer questions instantly, 24/7. A freelance consultant can                             │
│  use AI to draft proposals in a fraction of the time.                                                           │
│                                                                                                                 │
│  The businesses that adopt AI early aren't just saving time — they're freeing up                                │
│  mental energy to focus on what actually grows a business: relationships, creativity,                           │
│  and strategy. The tools will only get cheaper and smarter. The question isn't whether                          │
│  to adopt AI, but how soon you start.                                                                           │
│                                                                                                                 │
│      - Casual, engaging, hook-driven opening line                                                               │
│      - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)                               │
│      - Short paragraphs (2-3 lines max) separated by blank lines, heavy but tasteful emoji use                  │
│      - End with a call-to-action (e.g. 'Save this for later 🔖')                                                │
│      - CRITICAL: Use 8-15 Instagram-native hashtags that are DIFFERENT from typical LinkedIn                    │
│        professional hashtags — favor casual/community tags (e.g. #SmallBizTips #AIforBusiness                   │
│        #EntrepreneurLife #WorkSmarter) over formal ones (e.g. avoid #AIEngineering, #TechCareers style tags)    │
│      - Must read distinctly different from LinkedIn/Twitter versions in both tone AND hashtags                  │
│      - Output must be ready to copy-paste directly into Instagram — no markdown syntax                          │
│  Agent: Instagram Caption Writer                                                                                │
│                                                         

🚀 Ready to give your small biz a super‑charged boost?  

AI isn’t just for the tech giants anymore—affordable tools are here, and they work wonders for local shops.  

No‑code, user‑friendly platforms mean you don’t need a PhD to automate. A bakery can schedule posts, a gym can answer questions 24/7, and a freelancer can draft proposals in minutes.  

Early adopters aren’t just saving time; they’re freeing mental energy to focus on relationships, creativity, and strategy.  

The tools will only get cheaper and smarter—so the real question is: how soon will you start?  

Save this for later 🔖  

#SmallBizTips #AIforBiz #EntrepreneurLife #WorkSmarter #BizGrowth #DigitalSidekick #CreativeHustle #ShopLocal #BizHack #TechMadeEasy #AIForAll #StartupVibes #MarketingMagic #TimeSaver


### 9. Execution
We kicked off the CrewAI process asynchronously to generate the content based on the provided source text.

In [10]:
!pip install streamlit -q

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 219acca6-b826-4911-988a-de371e9fa899                                                                       │
│  Final Output: 🚀 Ready to give your small biz a super‑charged boost?                                           │
│                                                                                                                 │
│  AI isn’t just for the tech giants anymore—affordable tools are here, and they work wonders for local shops.    │
│                                                                                                                 │
│  No‑code, user‑friendly platforms mean you don’t need a PhD to automate. A bakery can schedule posts, a gym     │
│  can answer questions 24/7, and a freelancer can draft proposals in minutes.                                    │
│                                                                                                                 │
│  Early adopters aren’t just saving time; they’re freeing mental energy to focus on relationships, creativity,   │
│  and strategy.                                                                                                  │
│                                                                                                                 │
│  The tools will only get cheaper and smarter—so the real question is: how soon will you start?                  │
│                                                                                                                 │
│  Save this for later 🔖                                                                                         │
│                                                                                                                 │
│  #SmallBizTips #AIforBiz #EntrepreneurLife #WorkSmarter #BizGrowth #DigitalSidekick #CreativeHustle #ShopLocal  │
│  #BizHack #TechMadeEasy #AIForAll #StartupVibes #MarketingMagic #TimeSaver                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 44.0 MB/s eta 0:00:00


### 10. UI Installation
We installed `streamlit`, which allows us to turn this Python logic into a web-based application interface.

In [28]:
import re
import random

# ============================================
# STEP 1: Extract raw outputs
# ============================================
linkedin_raw = str(linkedin_task.output)
twitter_raw = str(twitter_task.output)
instagram_raw = str(instagram_task.output)


# ============================================
# STEP 2: Clean preamble + quote-wrapping
# ============================================
def clean_output(text):
    text = text.strip()
    text = re.sub(r'^(Here\'s|Here is|Post:|Below is|The rewritten).*?[:\n]+\s*',
                   '', text, flags=re.IGNORECASE)
    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1].strip()
    return text.strip()

linkedin_clean = clean_output(linkedin_raw)
twitter_clean = clean_output(twitter_raw)
instagram_clean = clean_output(instagram_raw)


# ============================================
# STEP 3: Hashtag extraction + AUTO-FIX for Instagram overlap
# ============================================
def get_hashtags(text):
    return re.findall(r'#\w+', text)

# Fallback pool of generic, casual, Instagram-style tags (used only if a
# duplicate needs replacing and we run out of obvious alternatives)
FALLBACK_IG_TAGS = [
    "#DailyMotivation", "#SelfImprovement", "#GrowthMindset", "#LifeHacks",
    "#MindsetShift", "#BetterEveryday", "#SmallWinsBigChange", "#RealTalk",
    "#LevelUp", "#ConsistencyIsKey", "#GoodHabits", "#PersonalGrowth"
]

def fix_instagram_hashtags(instagram_text, linkedin_text):
    li_tags = set(tag.lower() for tag in get_hashtags(linkedin_text))
    ig_tags = get_hashtags(instagram_text)

    fixed_tags = []
    used = set(t.lower() for t in ig_tags)
    fallback_pool = [t for t in FALLBACK_IG_TAGS if t.lower() not in li_tags]
    random.shuffle(fallback_pool)

    for tag in ig_tags:
        if tag.lower() in li_tags:
            # duplicate found — replace with a fresh fallback tag
            if fallback_pool:
                new_tag = fallback_pool.pop()
                while new_tag.lower() in used:
                    if not fallback_pool:
                        break
                    new_tag = fallback_pool.pop()
                fixed_tags.append(new_tag)
                used.add(new_tag.lower())
            else:
                fixed_tags.append(tag)  # no fallback left, keep as-is
        else:
            fixed_tags.append(tag)

    # Replace hashtag block in the original text
    # (assumes hashtags are all together at the end, which is the required format)
    text_without_tags = re.sub(r'(\s*#\w+)+\s*$', '', instagram_text).strip()
    new_text = text_without_tags + "\n\n" + " ".join(fixed_tags)
    return new_text, ig_tags, fixed_tags

instagram_fixed, old_ig_tags, new_ig_tags = fix_instagram_hashtags(instagram_clean, linkedin_clean)

print("=== HASHTAG AUTO-FIX REPORT ===")
print("LinkedIn tags:   ", get_hashtags(linkedin_clean))
print("Instagram before:", old_ig_tags)
print("Instagram after: ", new_ig_tags)


# ============================================
# STEP 4: Twitter word-count validator
# ============================================
def validate_tweets(thread_text):
    tweets = re.split(r'\n\s*(?=\d+/\d+)', thread_text.strip())
    print("\n=== TWITTER WORD COUNT CHECK ===")
    for tweet in tweets:
        tweet = tweet.strip()
        if not tweet:
            continue
        num_match = re.match(r'(\d+/\d+)', tweet)
        tweet_num = num_match.group(1) if num_match else "?"
        body = re.sub(r'^\d+/\d+:?\s*', '', tweet)
        body = re.sub(r'#\w+', '', body)
        word_count = len(body.split())
        status = " " if word_count <= 35 else " OVER LIMIT"
        print(f"{status} Tweet {tweet_num}: {word_count} words")

validate_tweets(twitter_clean)


# ============================================
# STEP 5: LinkedIn question-ending check
# ============================================
def ends_with_question(text):
    text_no_tags = re.sub(r'(\s*#\w+)+\s*$', '', text).strip()
    return text_no_tags.endswith('?')

print("\n=== LINKEDIN QUESTION-ENDING CHECK ===")
print(" Ends with question" if ends_with_question(linkedin_clean) else " Does NOT end with a question")


# ============================================
# STEP 6: Final outputs
# ============================================
print("\n" + "="*50)
print("FINAL OUTPUTS (Streamlit-ready)")
print("="*50)
print("\n--- LINKEDIN ---\n", linkedin_clean)
print("\n--- TWITTER ---\n", twitter_clean)
print("\n--- INSTAGRAM (hashtags auto-fixed) ---\n", instagram_fixed)

=== HASHTAG AUTO-FIX REPORT ===
LinkedIn tags:    ['#ArtificialIntelligence', '#SmallBusiness', '#DigitalTransformation', '#NoCode', '#GrowthMindset']
Instagram before: ['#SmallBizTips', '#AIforBiz', '#EntrepreneurLife', '#WorkSmarter', '#BizGrowth', '#DigitalSidekick', '#CreativeHustle', '#ShopLocal', '#BizHack', '#TechMadeEasy', '#AIForAll', '#StartupVibes', '#MarketingMagic', '#TimeSaver']
Instagram after:  ['#SmallBizTips', '#AIforBiz', '#EntrepreneurLife', '#WorkSmarter', '#BizGrowth', '#DigitalSidekick', '#CreativeHustle', '#ShopLocal', '#BizHack', '#TechMadeEasy', '#AIForAll', '#StartupVibes', '#MarketingMagic', '#TimeSaver']

=== TWITTER WORD COUNT CHECK ===
  Tweet 1/5: 20 words
  Tweet 2/5: 19 words
  Tweet 3/5: 22 words
  Tweet 4/5: 19 words
  Tweet 5/5: 18 words

=== LINKEDIN QUESTION-ENDING CHECK ===
 Ends with question

FINAL OUTPUTS (Streamlit-ready)

--- LINKEDIN ---
 Small businesses are missing out if they ignore AI today.

Artificial intelligence is no longer the exc

In [29]:
import os
from google.colab import userdata

groq_key = userdata.get('GROQ_API_KEY')

app_code = rf'''
import streamlit as st
import os
import re
import random
from crewai import Agent, Task, Crew, Process, LLM
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

st.set_page_config(page_title="Social Media Content Generator", layout="wide")
st.title(" AI Social Media Content Generator")
st.caption("CrewAI + Groq | LinkedIn / Twitter / Instagram")

os.environ["GROQ_API_KEY"] = "{groq_key}"

source_content = st.text_area("Enter your source content:", height=300, placeholder="Paste your article or notes here...")

if st.button("Generate Social Posts", type="primary"):
    if not source_content.strip():
        st.warning("Please enter some content first.")
    else:
        with st.spinner("Agents are working on your content..."):

            # ============================================
            # LLM (same model/temperature as tested in notebook)
            # ============================================
            llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.7)

            # ============================================
            # AGENTS
            # ============================================
            linkedin_writer = Agent(
                role="LinkedIn Content Writer",
                goal="Turn the source content into a professional, engaging LinkedIn post with a hook, value, and a call-to-action",
                backstory=(
                    "You are an expert LinkedIn ghostwriter who knows how to write posts that "
                    "get engagement — short paragraphs, a strong opening line, and relevant hashtags."
                ),
                llm=llm,
                verbose=True
            )

            twitter_writer = Agent(
                role="Twitter/X Thread Writer",
                goal="Turn the source content into a concise, punchy Twitter/X thread (3-6 tweets)",
                backstory=(
                    "You are a Twitter/X growth expert who writes threads that hook readers "
                    "in the first tweet and keep them scrolling."
                ),
                llm=llm,
                verbose=True
            )

            instagram_writer = Agent(
                role="Instagram Caption Writer",
                goal="Turn the source content into a catchy Instagram caption with relevant hashtags and emojis",
                backstory=(
                    "You are a social media copywriter specializing in Instagram captions that "
                    "are visually scannable, emotionally engaging, and hashtag-optimized."
                ),
                llm=llm,
                verbose=True
            )

            # ============================================
            # TASKS (same rules tested and validated in the notebook)
            # ============================================
            linkedin_task = Task(
                description=f"""Rewrite the blog content as a LinkedIn post.
                Source Content: {{source_content}}
                - Professional, thought-leadership tone
                - 150-250 words
                - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols like * or -)
                - Use line breaks (blank lines) for readability instead of headers
                - MANDATORY: The post MUST end with a genuine question (not a statement or CTA) —
                  verify the last sentence is a question before finalizing
                - Add 3-5 professional hashtags at the very end, space-separated (e.g. #AIEngineering #TechCareers)
                - Do NOT reuse blog's exact headings or sentence structure
                - CRITICAL: Output must be RAW plain text only — do NOT wrap the post in quotation marks,
                  do NOT prefix it with "Post:" or any label. Start directly with the hook line
                - Output must be ready to copy-paste directly into LinkedIn — no markdown syntax""",
                expected_output="Plain text LinkedIn post (no markdown, no quotes, no labels), 150-250 words, ending with a question, then hashtags.",
                agent=linkedin_writer,
                async_execution=True,
                context=[]
            )

            twitter_task = Task(
                description=f"""Convert the blog into a Twitter/X thread.
                Source Content: {{source_content}}
                - CRITICAL: Each tweet's text (excluding numbering/hashtags) MUST be 35 words or fewer — count before finalizing
                - Punchy, conversational tone — short sentences
                - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)
                - Use thread numbering like "1/5" at the start of each tweet
                - CRITICAL: Distribute hashtags ACROSS the thread — do NOT dump all hashtags on the last tweet.
                  Each tweet should carry 0-2 of its own hashtags, spread naturally through the thread
                - Rephrase content in your own words
                - Separate each tweet clearly with a blank line so it's easy to copy individually""",
                expected_output="Plain text Twitter thread, numbered tweets separated by blank lines, each under 35 words, hashtags distributed across tweets not stacked on one.",
                agent=twitter_writer,
                async_execution=True,
                context=[]
            )

            instagram_task = Task(
                description=f"""Write an Instagram caption based on the blog.
                Source Content: {{source_content}}
                - Casual, engaging, hook-driven opening line
                - Plain text only — NO markdown formatting (no **, no ##, no bullet symbols)
                - Short paragraphs (2-3 lines max) separated by blank lines, heavy but tasteful emoji use
                - End with a call-to-action (e.g. 'Save this for later 🔖')
                - CRITICAL: Use 8-15 Instagram-native hashtags that are DIFFERENT from typical LinkedIn
                  professional hashtags — favor casual/community tags (e.g. #SmallBizTips #DailyMotivation
                  #EntrepreneurLife #WorkSmarter) over formal ones
                - Must read distinctly different from LinkedIn/Twitter versions in both tone AND hashtags
                - Output must be ready to copy-paste directly into Instagram — no markdown syntax""",
                expected_output="Plain text Instagram caption (no markdown), short paragraphs, ending with hashtags that differ from LinkedIn's hashtags.",
                agent=instagram_writer,
                async_execution=False,
                context=[]
            )

            crew = Crew(
                agents=[linkedin_writer, twitter_writer, instagram_writer],
                tasks=[linkedin_task, twitter_task, instagram_task],
                process=Process.sequential,
                verbose=True
            )

            result = crew.kickoff()

            # ============================================
            # POST-PROCESSING / VALIDATION PIPELINE
            # (same logic tested in the notebook)
            # ============================================
            linkedin_raw = str(linkedin_task.output)
            twitter_raw = str(twitter_task.output)
            instagram_raw = str(instagram_task.output)

            def clean_output(text):
                text = text.strip()
                text = re.sub(r"^(Here\'s|Here is|Post:|Below is|The rewritten).*?[:\n]+\s*",
                               "", text, flags=re.IGNORECASE)
                if (text.startswith(chr(34)) and text.endswith(chr(34))) or (text.startswith("\'") and text.endswith("\'")):
                    text = text[1:-1].strip()
                return text.strip()

            linkedin_clean = clean_output(linkedin_raw)
            twitter_clean = clean_output(twitter_raw)
            instagram_clean = clean_output(instagram_raw)

            def get_hashtags(text):
                return re.findall(r"#\w+", text)

            FALLBACK_IG_TAGS = [
                "#DailyMotivation", "#SelfImprovement", "#GrowthMindset", "#LifeHacks",
                "#MindsetShift", "#BetterEveryday", "#SmallWinsBigChange", "#RealTalk",
                "#LevelUp", "#ConsistencyIsKey", "#GoodHabits", "#PersonalGrowth"
            ]

            def fix_instagram_hashtags(instagram_text, linkedin_text):
                li_tags = set(tag.lower() for tag in get_hashtags(linkedin_text))
                ig_tags = get_hashtags(instagram_text)

                fixed_tags = []
                used = set(t.lower() for t in ig_tags)
                fallback_pool = [t for t in FALLBACK_IG_TAGS if t.lower() not in li_tags]
                random.shuffle(fallback_pool)

                for tag in ig_tags:
                    if tag.lower() in li_tags:
                        if fallback_pool:
                            new_tag = fallback_pool.pop()
                            while new_tag.lower() in used:
                                if not fallback_pool:
                                    break
                                new_tag = fallback_pool.pop()
                            fixed_tags.append(new_tag)
                            used.add(new_tag.lower())
                        else:
                            fixed_tags.append(tag)
                    else:
                        fixed_tags.append(tag)

                text_without_tags = re.sub(r"(\s*#\w+)+\s*$", "", instagram_text).strip()
                new_text = text_without_tags + chr(10) + chr(10) + " ".join(fixed_tags)
                return new_text, ig_tags, fixed_tags

            instagram_fixed, old_ig_tags, new_ig_tags = fix_instagram_hashtags(instagram_clean, linkedin_clean)

            def validate_tweets(thread_text):
                tweets = re.split(r"\n\s*(?=\d+/\d+)", thread_text.strip())
                report = []
                for tweet in tweets:
                    tweet = tweet.strip()
                    if not tweet:
                        continue
                    num_match = re.match(r"(\d+/\d+)", tweet)
                    tweet_num = num_match.group(1) if num_match else "?"
                    body = re.sub(r"^\d+/\d+:?\s*", "", tweet)
                    body = re.sub(r"#\w+", "", body)
                    word_count = len(body.split())
                    ok = word_count <= 35
                    report.append((tweet_num, word_count, ok))
                return report

            tweet_report = validate_tweets(twitter_clean)

            def ends_with_question(text):
                text_no_tags = re.sub(r"(\s*#\w+)+\s*$", "", text).strip()
                return text_no_tags.endswith("?")

            li_question_ok = ends_with_question(linkedin_clean)

            # ============================================
            # DISPLAY
            # ============================================
            st.divider()
            st.header(" Generated Content")

            st.subheader(" LinkedIn Post")
            st.text_area("LinkedIn", value=linkedin_clean, height=280, label_visibility="collapsed")
            if not li_question_ok:
                st.caption(" Note: post may not end with a question — review before posting.")

            st.divider()
            st.subheader(" Twitter/X Thread")
            st.text_area("Twitter", value=twitter_clean, height=280, label_visibility="collapsed")
            over_limit = [t for t in tweet_report if not t[2]]
            if over_limit:
                over_str = ", ".join(f"{{t[0]}} ({{t[1]}}w)" for t in over_limit)
                st.caption(f" Tweets over 35 words: {{over_str}}")

            st.divider()
            st.subheader(" Instagram Caption")
            st.text_area("Instagram", value=instagram_fixed, height=280, label_visibility="collapsed")

            with st.expander("Validation details"):
                st.write("LinkedIn hashtags:", get_hashtags(linkedin_clean))
                st.write("Instagram hashtags (before fix):", old_ig_tags)
                st.write("Instagram hashtags (after fix):", new_ig_tags)
                st.write("Tweet word counts:", tweet_report)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('app.py updated: synced with notebook-tested agents, tasks, model, and validation pipeline!')


app.py updated: synced with notebook-tested agents, tasks, model, and validation pipeline!


### 11. Application Development (`app.py`)
We wrote the `app.py` file. This script contains the Streamlit UI code and an optimized CrewAI workflow (using the Manager role) that uses subheaders and dividers to organize the output for LinkedIn, Twitter, and Instagram clearly.

In [31]:
!pip install pyngrok -q

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))


### 12. Tunneling Setup
We installed and configured `pyngrok` to create a secure tunnel, making the local Streamlit server accessible via a public URL.

In [32]:
import subprocess
import time
from pyngrok import ngrok

# Terminate existing sessions
!pkill -f streamlit
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
except:
    pass

# Restart Streamlit
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

# Reconnect tunnel
public_url = ngrok.connect(8501)
print("Your organized app is live at:", public_url)

Your organized app is live at: NgrokTunnel: "https://postnasal-angled-sessions.ngrok-free.dev" -> "http://localhost:8501"


### 13. Launching the App
In this final step, we stopped any old instances, started the Streamlit server in the background, and generated the live ngrok URL for the user to access the application.